In [6]:
import json
from pathlib import Path
from typing import Any

import torch
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    pipeline,
)


# --------------------------------------------------
# Configuration
# --------------------------------------------------


MODEL_NAME = "Wismut/nym-pii-multilingual"
CONFIDENCE_THRESHOLD = 0.50


# --------------------------------------------------
# Load the model
# --------------------------------------------------

device = 0 if torch.cuda.is_available() else -1

print(f"Loading model: {MODEL_NAME}")
print(f"Device: {'GPU' if device == 0 else 'CPU'}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

entity_detector = pipeline(
    task="token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=device,
)


# --------------------------------------------------
# Entity extraction
# --------------------------------------------------

def extract_entities_from_text(
    text: str,
    json_path: str,
) -> list[dict[str, Any]]:
    """
    Detect PII entities in a single text value.

    The detected start/end offsets are relative to this particular
    JSON string value.
    """

    if not text.strip():
        return []

    # Do not pass truncation=True here.
    # TokenClassificationPipeline does not support it in some
    # Transformers versions.
    predictions = entity_detector(text)

    entities = []

    for prediction in predictions:
        confidence = float(prediction["score"])

        if confidence < CONFIDENCE_THRESHOLD:
            continue

        entity_text = prediction["word"]
        start = int(prediction["start"])
        end = int(prediction["end"])

        # Prefer the exact substring from the source text. This avoids
        # tokenizer artifacts such as leading spaces or subword markers.
        if 0 <= start < end <= len(text):
            entity_text = text[start:end]

        entities.append(
            {
                "json_path": json_path,
                "entity": entity_text,
                "entity_type": prediction["entity_group"],
                "confidence": round(confidence, 4),
                "start": start,
                "end": end,
            }
        )

    return entities


# --------------------------------------------------
# Recursive JSON traversal
# --------------------------------------------------

def extract_entities_from_json(
    value: Any,
    json_path: str = "$",
) -> list[dict[str, Any]]:
    """
    Recursively inspect all string values in dictionaries and lists.
    """

    entities = []

    if isinstance(value, dict):
        for key, child_value in value.items():
            child_path = f"{json_path}.{key}"

            entities.extend(
                extract_entities_from_json(
                    value=child_value,
                    json_path=child_path,
                )
            )

    elif isinstance(value, list):
        for index, child_value in enumerate(value):
            child_path = f"{json_path}[{index}]"

            entities.extend(
                extract_entities_from_json(
                    value=child_value,
                    json_path=child_path,
                )
            )

    elif isinstance(value, str):
        entities.extend(
            extract_entities_from_text(
                text=value,
                json_path=json_path,
            )
        )

    return entities


# --------------------------------------------------
# File processing
# --------------------------------------------------

def process_json_file(
    input_path: str,
    output_path: str | None = None,
) -> list[dict[str, Any]]:
    input_file = Path(input_path)

    if not input_file.exists():
        raise FileNotFoundError(
            f"JSON file not found: {input_file.resolve()}"
        )

    if not input_file.is_file():
        raise ValueError(
            f"Input path is not a file: {input_file.resolve()}"
        )

    with input_file.open(
        mode="r",
        encoding="utf-8",
    ) as file:
        json_data = json.load(file)

    entities = extract_entities_from_json(json_data)

    if output_path:
        output_file = Path(output_path)

        output_file.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        with output_file.open(
            mode="w",
            encoding="utf-8",
        ) as file:
            json.dump(
                entities,
                file,
                ensure_ascii=False,
                indent=2,
            )

    return entities


# --------------------------------------------------
# Run
# --------------------------------------------------

if __name__ == "__main__":
    JSON_FILE_PATH = r"input.json"
    OUTPUT_FILE_PATH = r"detected_entities.json"

    detected_entities = process_json_file(
        input_path=JSON_FILE_PATH,
        output_path=OUTPUT_FILE_PATH,
    )

    print(
        json.dumps(
            detected_entities,
            ensure_ascii=False,
            indent=2,
        )
    )

    print(f"\nDetected {len(detected_entities)} entities.")
    print(f"Results saved to: {OUTPUT_FILE_PATH}")



Loading model: Wismut/nym-pii-multilingual
Device: GPU


Loading weights: 100%|██████████| 138/138 [00:00<00:00, 11739.70it/s]


[
  {
    "json_path": "$.customer.name",
    "entity": "John",
    "entity_type": "GIVEN_NAME",
    "confidence": 0.9999,
    "start": 0,
    "end": 4
  },
  {
    "json_path": "$.customer.name",
    "entity": " Smith",
    "entity_type": "SURNAME",
    "confidence": 0.9965,
    "start": 4,
    "end": 10
  },
  {
    "json_path": "$.customer.email",
    "entity": "john.smith@example.com",
    "entity_type": "EMAIL",
    "confidence": 1.0,
    "start": 0,
    "end": 22
  },
  {
    "json_path": "$.customer.phone",
    "entity": "+33 6 12 34 56 78",
    "entity_type": "PHONE",
    "confidence": 1.0,
    "start": 0,
    "end": 17
  },
  {
    "json_path": "$.notes[0]",
    "entity": "10",
    "entity_type": "BUILDING_NUMBER",
    "confidence": 0.6035,
    "start": 21,
    "end": 23
  },
  {
    "json_path": "$.notes[0]",
    "entity": " Avenue des Champs-Élysées",
    "entity_type": "STREET_NAME",
    "confidence": 0.9461,
    "start": 23,
    "end": 49
  },
  {
    "json_path": "$.notes